In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import numpy as np, pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/lung-ct/crops")
meta2 = pd.read_csv("/content/drive/MyDrive/lung-ct/crops_meta_split.csv")

arrays = {}
for pid in meta2["dicom_patient"].unique():
    z = np.load(OUT / f"{pid}.npz")
    for k in z.files:
        arrays[k] = z[k]

missing = [k for k in meta2["key"] if k not in arrays]
print(len(arrays), len(meta2), "brakujących:", len(missing))
print(round(sum(a.nbytes for a in arrays.values()) / 1e6), "MB")

20339 20236 brakujących: 0
637 MB


In [5]:
import torch
import torch.nn.functional as F

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
AIR = (-1000 + 1400) / 2100   # powietrze w skali okna

def preprocess(arr, size=224):
    x = (np.clip(arr.astype(np.float32), -1400, 700) + 1400) / 2100
    h, w = x.shape
    s = max(h, w)
    sq = np.full((s, s), AIR, np.float32)
    y0, x0 = (s - h) // 2, (s - w) // 2
    sq[y0:y0 + h, x0:x0 + w] = x
    t = torch.from_numpy(sq)[None, None]
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)[0]
    t = t.repeat(3, 1, 1)
    return (t - MEAN) / STD

r = meta2.iloc[0]
x = preprocess(arrays[r["key"]])
print(x.shape, float(x.min()), float(x.max()))

torch.Size([3, 224, 224]) -1.3360366821289062 2.640000104904175


In [6]:
from torch.utils.data import Dataset, DataLoader

CLASSES = ["A", "B", "E", "G"]
cls2id = {c: i for i, c in enumerate(CLASSES)}

class CropDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return preprocess(arrays[r["key"]]), cls2id[r["cls"]]

dfs = {s: meta2[meta2["split"] == s] for s in ["train", "val", "test"]}
loaders = {
    s: DataLoader(CropDS(d), batch_size=32, shuffle=(s == "train"), num_workers=2)
    for s, d in dfs.items()
}

xb, yb = next(iter(loaders["train"]))
print(xb.shape, yb[:12].tolist())
print({s: len(d) for s, d in dfs.items()})

torch.Size([32, 3, 224, 224]) [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 3]
{'train': 13306, 'val': 3328, 'test': 3602}


In [7]:
import torchvision

device = "cuda"
print(torch.cuda.get_device_name(0))

model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

out = model(xb.to(device))
loss = torch.nn.functional.cross_entropy(out, yb.to(device))
print(out.shape, round(loss.item(), 3))

Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


torch.Size([32, 4]) 1.512


In [8]:
import time

opt = torch.optim.Adam(model.parameters(), lr=1e-4)
model.train()
xg, yg = xb.to(device), yb.to(device)

t = time.time()
for step in range(30):
    opt.zero_grad()
    loss = F.cross_entropy(model(xg), yg)
    loss.backward()
    opt.step()
    if step % 5 == 0:
        print(step, round(loss.item(), 3))
torch.cuda.synchronize()
print("s/krok:", round((time.time() - t) / 30, 3))

0 1.512
5 0.106
10 0.014
15 0.004
20 0.002
25 0.001
s/krok: 0.305


In [9]:
from sklearn.metrics import f1_score

def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return f1_score(y, p, average="macro"), (y == p).mean()

def run(model, params, lr, epochs, weight=None, freeze_bn=False):
    opt = torch.optim.Adam(params, lr=lr)
    for ep in range(epochs):
        model.train()
        if freeze_bn:
            for m in model.modules():
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eval()
        t, tot = time.time(), 0
        for x, y in loaders["train"]:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y, weight=weight)
            loss.backward(); opt.step()
            tot += loss.item() * len(y)
        f1, acc = evaluate(model, loaders["val"])
        print(f"ep{ep+1} loss {tot/len(dfs['train']):.3f} val macroF1 {f1:.3f} acc {acc:.3f} ({time.time()-t:.0f}s)")

model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True

run(model, model.fc.parameters(), 1e-3, 3, freeze_bn=True)

ep1 loss 0.689 val macroF1 0.346 acc 0.696 (58s)
ep2 loss 0.566 val macroF1 0.389 acc 0.726 (62s)
ep3 loss 0.518 val macroF1 0.365 acc 0.648 (61s)


In [10]:
for p in model.parameters():
    p.requires_grad = True

run(model, model.parameters(), 1e-5, 5)

ep1 loss 0.416 val macroF1 0.377 acc 0.687 (158s)
ep2 loss 0.099 val macroF1 0.375 acc 0.685 (157s)
ep3 loss 0.031 val macroF1 0.368 acc 0.690 (157s)
ep4 loss 0.014 val macroF1 0.377 acc 0.714 (156s)
ep5 loss 0.007 val macroF1 0.359 acc 0.694 (156s)


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval(); ys, ps = [], []
with torch.no_grad():
    for x, y in loaders["val"]:
        ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
print(confusion_matrix(y, p))
print(classification_report(y, p, target_names=CLASSES, digits=3, zero_division=0))

[[1937  152    0  536]
 [  65   30    0   41]
 [   4    0    0    0]
 [ 176   44    0  343]]
              precision    recall  f1-score   support

           A      0.888     0.738     0.806      2625
           B      0.133     0.221     0.166       136
           E      0.000     0.000     0.000         4
           G      0.373     0.609     0.463       563

    accuracy                          0.694      3328
   macro avg      0.348     0.392     0.359      3328
weighted avg      0.769     0.694     0.721      3328



In [12]:
def make_model():
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = torch.nn.Linear(m.fc.in_features, len(CLASSES))
    return m.to(device)

def experiment(weight=None, ep1=3, ep2=3):
    m = make_model()
    for p in m.parameters():
        p.requires_grad = False
    for p in m.fc.parameters():
        p.requires_grad = True
    run(m, m.fc.parameters(), 1e-3, ep1, weight, freeze_bn=True)
    for p in m.parameters():
        p.requires_grad = True
    run(m, m.parameters(), 1e-5, ep2, weight)
    return m

counts = dfs["train"]["cls"].value_counts().reindex(CLASSES).values
w = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32, device=device)
print(w)
model_w = experiment(w)

tensor([ 0.3746,  2.4140, 18.6882,  1.1587], device='cuda:0')
ep1 loss 0.939 val macroF1 0.275 acc 0.441 (63s)
ep2 loss 0.715 val macroF1 0.309 acc 0.487 (61s)
ep3 loss 0.628 val macroF1 0.327 acc 0.528 (61s)
ep1 loss 0.552 val macroF1 0.351 acc 0.607 (157s)
ep2 loss 0.178 val macroF1 0.360 acc 0.624 (157s)
ep3 loss 0.073 val macroF1 0.362 acc 0.641 (157s)
